# Test Local LLM Setup

Verify that your local LLM and embedding servers are working correctly.

In [27]:
import requests
import json
from openai import OpenAI

## 1. Test LLM Server

In [29]:
# Test chat completion
llm_client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="sk-local-llm-key"
)

response = llm_client.chat.completions.create(
    model="gemma-4-e4b",
    messages=[
        {"role": "user", "content": "שלום! מה דעתך על הצעות חוק בכנסת?"}
    ],
    temperature=0.7
)

print("Response:", response.choices[0].message.content)

Response: זו שאלה רחבה ומורכבת מאוד, והתשובה עליה תלויה לחלוטין באיזה חוק מדובר ובאיזה היבט של החוק מעניין אותך.

בתור מודל בינה מלאכותית, אין לי דעה אישית, פוליטית או אידיאולוגית. לכן, במקום לתת לך "דעה" על החוקים, אני יכול לספק לך **ניתוח, הקשר, וכלים** שיעזרו לך להבין את ההצעות הללו מכל זווית אפשרית.

**כדי שנוכל לנהל שיח מועיל, חשוב להבין מהי הגישה שלך:**

---

### 🏛️ אם אתה מעוניין בניתוח משפטי/פוליטי:

אני יכול לעזור לך לבחון:

1.  **ההשלכות המשפטיות:** האם החוק עומד במבחני החוקה? האם הוא יוצר סתירות עם חקיקה קיימת?
2.  **ההשלכות החברתיות:** איך החוק ישפיע על קבוצות שונות באוכלוסייה (כלכלה, זכויות אדם, מגדר וכו')?
3.  **נקודות המחלוקת המדיניות:** מהן הטיעונים של המפלגות השונות (ימין, שמאל, מרכז)? מה היתרונות והחסרונות שמציגים כל הצדדים?
4.  **התהליך החקיקתי:** האם החוק עובר תהליך מסודר? האם הוא מוגן מפני השפעות פוליטיות יתר?

### ⚖️ אם אתה מעוניין בהשוואה בינלאומית:

אני יכול לבדוק:

*   האם מדינות אחרות (בתוך או מחוץ לישראל) אימצו חקיקה דומה.
*   מהם היתרונות והחסרונות של גישות 

## 2. Test Embedding Server

In [30]:
# Test embedding generation
embed_client = OpenAI(
    base_url="http://localhost:8081/v1",
    api_key="sk-local-llm-key"
)

texts = [
    "הצעת חוק לתיקון סעיף 5 בחוק הכנסת",
    "הצעת חוק להגברת השקיפות בממשלה"
]

response = embed_client.embeddings.create(
    model="multilingual-e5-large",
    input=texts
)

print(f"Generated {len(response.data)} embeddings")
print(f"Embedding dimension: {len(response.data[0].embedding)}")
print(f"First embedding (first 5 values): {response.data[0].embedding[:5]}")

Generated 2 embeddings
Embedding dimension: 768
First embedding (first 5 values): [-0.012520665302872658, 0.05210160091519356, -0.13467204570770264, -0.036616042256355286, 0.01371803879737854]


## 3. Test Semantic Similarity

In [31]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Get embeddings for two texts
texts = [
    "הצעת חוק להגברת השקיפות בממשלה",
    "הצעת חוק לפיקוח על הממשלה",
    "הצעת חוק לשיפור התחבורה הציבורית"
]

response = embed_client.embeddings.create(
    model="multilingual-e5-large",
    input=texts
)

embeddings = [d.embedding for d in response.data]

print("Similarity matrix:")
for i, text_i in enumerate(texts):
    for j, text_j in enumerate(texts):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f"  [{i}] vs [{j}]: {sim:.3f}")
    print()

Similarity matrix:
  [0] vs [0]: 1.000
  [0] vs [1]: 0.924
  [0] vs [2]: 0.917

  [1] vs [0]: 0.924
  [1] vs [1]: 1.000
  [1] vs [2]: 0.890

  [2] vs [0]: 0.917
  [2] vs [1]: 0.890
  [2] vs [2]: 1.000



## 4. Test with Real Bill Data

In [32]:
import pandas as pd

# Load your bills dataset
df = pd.read_excel('../dataset/חיפוש_הצעות_חוק_1_8_2026.xlsx')

# Take a sample bill
sample_bill = df.iloc[0]
bill_name = sample_bill['שם']

print("Bill title:", bill_name)
print("\nTesting LLM summarization...")

# Test LLM summarization
response = llm_client.chat.completions.create(
    model="gemma-4-e4b",
    messages=[
        {"role": "system", "content": "אתה עוזר שמסכם הצעות חוק בקצרה."},
        {"role": "user", "content": f"סכם במשפט אחד: {bill_name}"}
    ],
    temperature=0.5
)

print("\n=== Full response ===")
print(response)
print("\n=== Message content ===")
content = response.choices[0].message.content
print(f"Type: {type(content)}")
print(f"Length: {len(content) if content else 0}")
print(f"Value: {repr(content)}")
print(f"Stripped: {repr(content.strip()) if content else None}")
print("\n=== Usage ===")
print(response.usage)

Bill title: חוק לתיקון פקודת בתי הסוהר (הארכת הוראות שעה) (תיקוני חקיקה), התשפ"ו-2026

Testing LLM summarization...

=== Full response ===
ChatCompletion(id='chatcmpl-wILfy4gZKSMahNzCkRGDwVfdpCMwsZZJ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='החוק מאריך את תוקף ההוראות הזמניות הקיימות בפקודת בתי הסוהר.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content='Here\'s a thinking process to arrive at the suggested summary:\n\n1.  **Analyze the Request:** The user wants a one-sentence summary of a specific law: "חוק לתיקון פקודת בתי הסוהר (הארכת הוראות שעה) (תיקוני חקיקה), התשפ"ו-2026" (Law to Amend the Prisons Ordinance (Extension of Temporary Provisions) (Legislative Amendments), 2026).\n2.  **Deconstruct the Law\'s Title (Keywords):**\n    *   *חוק לתיקון פקודת בתי הסוהר* (Law to Amend the Prisons Ordinance): The core subject is the prison system.\n    *   *(הארכת ה

In [ ]:
# Debug: Test different prompts to see what works
test_prompts = [
    "סכם את הטקסט הבא: חוק חדש לתיקון פקודת בתי הסוהר",
    "מה דעתך על הצעת חוק זו?",
    "תן לי תקציר קצר של: חוק לתיקון פקודת בתי הסוהר"
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n=== Test {i} ===")
    print(f"Prompt: {prompt}")
    
    response = llm_client.chat.completions.create(
        model="gemma-4-e4b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
        max_tokens=100
    )
    
    content = response.choices[0].message.content
    print(f"Response: {repr(content)}")
    print(f"Tokens: {response.usage.completion_tokens}")

In [ ]:
import time

# Measure LLM response time
start = time.time()
response = llm_client.chat.completions.create(
    model="gemma-4-e4b",
    messages=[{"role": "user", "content": "חזור על זה: שלום עולם"}],
    max_tokens=20
)
llm_time = time.time() - start

print(f"LLM response time: {llm_time:.2f}s")
print(f"Tokens generated: {response.usage.completion_tokens}")
print(f"Tokens/sec: {response.usage.completion_tokens / llm_time:.1f}")

LLM response time: 1.01s
Tokens generated: 5
Tokens/sec: 5.0


In [ ]:
# Measure embedding speed
n_docs = 10
texts = ["טקסט לדוגמה מספר " + str(i) for i in range(n_docs)]

start = time.time()
response = embed_client.embeddings.create(
    model="multilingual-e5-large",
    input=texts
)
embed_time = time.time() - start

print(f"Embedding time for {n_docs} docs: {embed_time:.2f}s")
print(f"Docs/sec: {n_docs / embed_time:.1f}")

Embedding time for 10 docs: 0.28s
Docs/sec: 35.6


## 6. Debug: Test Without System Prompt

In [ ]:
# Test without system prompt and with higher max_tokens
# Sometimes Gemma's thinking/reasoning uses tokens that don't appear in output

test_prompts = [
    ("With system prompt, max_tokens=100", [
        {"role": "system", "content": "אתה עוזר שמסכם הצעות חוק בקצרה."},
        {"role": "user", "content": f"סכם במשפט אחד: {bill_name}"}
    ], 100),
    ("Without system prompt, max_tokens=100", [
        {"role": "user", "content": f"סכם במשפט אחד: {bill_name}"}
    ], 100),
    ("Without system prompt, max_tokens=500", [
        {"role": "user", "content": f"סכם במשפט אחד: {bill_name}"}
    ], 500),
]

for title, messages, max_tok in test_prompts:
    print(f"\n=== {title} ===")
    response = llm_client.chat.completions.create(
        model="gemma-4-e4b",
        messages=messages,
        temperature=0.5,
        max_tokens=max_tok
    )
    content = response.choices[0].message.content
    print(f"Content: {repr(content[:100] if content else content)}")
    print(f"Tokens: prompt={response.usage.prompt_tokens}, completion={response.usage.completion_tokens}")
    print(f"Finish reason: {response.choices[0].finish_reason}")